In [1]:
import pandas as pd

from tqdm import tqdm
import json
import os
from collections import defaultdict
from ast import literal_eval

import torch
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer
import sys
# Add the directory to your path
sys.path.append('/home/sp3945/mod/ruler/')

from ruler.data.constants import URL_REPLACEMENT_TOKEN
from ruler.data.text_utils import markdown_to_text, replace_urls


In [ ]:
def augment_with_languages(text):
    model_ckpt = "papluca/xlm-roberta-base-language-detection"
    pipe = pipeline("text-classification", model=model_ckpt, device_map="auto")
    language = pipe(text, top_k=1, truncation=True)[0]['label']
    return language
    #descs_df['language'] = list(map(lambda x: x[0]['label'], pipe(descs_df.description.tolist(), top_k=1, truncation=True)))
    #return descs_df


In [ ]:
def predict_NuExtract(text,model, tokenizer,  schema, example=["", "", ""]):
    text = text.replace('"','').replace("'","")
    schema = json.dumps(json.loads(schema))
    input_llm =  "<|input|>\n### Template:\n" +  schema + "\n"
    for i in example:
      if i != "":
          input_llm += "### Example:\n"+ json.dumps(json.loads(i), indent=4)+"\n"
    
    input_llm +=  "### Text:\n"+text +"\n<|output|>\n"    
    try:
        input_ids = tokenizer(input_llm, return_tensors="pt", truncation=True, max_length=4000).to("cuda")
        with torch.no_grad():
            output = tokenizer.decode(model.generate(**input_ids, max_new_tokens=4000)[0], skip_special_tokens=True)
        return output.split("<|output|>")[1].split("<|end-output|>")[0]
    except Exception as e:
        print(f"Error during model inference: {e}")
        return ""


In [ ]:
data_dir = "../data"

description_fpath = os.path.join(data_dir, 'interim', 'community_descriptions.json')
with open(description_fpath, encoding='utf8') as f:
    descs = json.load(f)

descs_multiple = {k: v for k, v in descs.items() if len(v) > 1 and len(v) < 6}

In [ ]:
len(descs_multiple.keys())

In [ ]:
cache_dir = "/storage/shruti/huggingface/"
model_name = "numind/NuExtract-v1.5"
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir=cache_dir, torch_dtype=torch.bfloat16, trust_remote_code=True).to(
            device).eval()
tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir=cache_dir, trust_remote_code=True)
schema = """{
    "Rules": [{
    "Number": 1,
    "Description": ""
    }]
}"""

In [ ]:
community_meta = defaultdict(list)

for c in tqdm(descs_multiple.keys()):
    commlangs = set()
    for c_d in descs_multiple[c]:
        language = augment_with_languages(c_d)
        commlangs.add(language)
    if len(commlangs)==1 and list(commlangs)[0]=='en':
        for c_d in descs_multiple[c]:
            rules = predict_NuExtract(c_d,model=model, tokenizer=tokenizer,  schema=schema, example=["", "", ""])
            tempd = {}
            tempd['description'] = c_d
            tempd['language'] = 'en'
            tempd['rules'] = rules
            community_meta[c].append(tempd)
    else:
        print(c)

In [ ]:
with open("../data/interim/community_meta.json", "w") as jfile:
    json.dump(community_meta, jfile)

In [2]:
with open("../data/interim/community_meta.json", "r") as jfile:
    community_meta = json.load(jfile)

In [3]:
len(community_meta.keys())

501

In [4]:
from random import sample
sample(list(community_meta.keys()), 5)

['https://lemmy.world/c/rimworld',
 'https://feddit.uk/c/homevideo',
 'https://lemmy.world/c/cooking',
 'https://lemmy.world/c/gunners',
 'https://lemmynsfw.com/c/anal']

In [ ]:
community_meta['https://lemmy.world/c/mildlyinfuriating']

In [5]:
def convert_json_format(text):
    try:
        rule=json.loads(text)
        simplified_rules={}
        simplified_rules = {    "rules": {int(rul["Number"]): rul["Description"] for rul in rule["Rules"]}}
        return simplified_rules
    except Exception as e:
        return e


In [6]:
for c in community_meta.keys():
    # if len(community_meta[c]) > 1:
    for i in range(len(community_meta[c])):
        simplified_rules = convert_json_format(community_meta[c][i]['rules'])
        community_meta[c][i]['rules'] = simplified_rules

In [7]:
community_meta['https://lemmy.world/c/mildlyinfuriating']

[{'description': 'Home to all things &quot;Mildly Infuriating&quot;\nNot infuriating, not enraging. Mildly Infuriating. All posts should reflect that.\n\nI want my day mildly ruined, not completely ruined.\nPlease remember to refrain from reposting old content. If you post a post from reddit it is good practice to include a link and credit the OP. I&#x27;m not about stealing content!\n\nIt&#x27;s just good to get something in this website for casual viewing whilst refreshing original content is added overtime.\n\n_________________________\n\n**Rules:**\n\n::: spoiler 1. Be Respectful\n___\n\nRefrain from using harmful language pertaining to a protected characteristic: e.g. race, gender, sexuality, disability or religion.\n\nRefrain from being argumentative when responding or commenting to posts/replies. Personal attacks are not welcome here.\n\n...\n\n:::\n\n_________________________\n::: spoiler 2. No Illegal Content\n___\n\nContent that violates the law. Any post/comment found to be 

In [8]:
all_multi_comms = list(set(list(community_meta.keys())))
with open("../data/interim/filtered_communities.jsonl", "r") as jlfile:
    for line in jlfile:
        job = json.loads(line)
        if job['actor_id'] not in all_multi_comms:
            community_meta[job['actor_id']] = []
            tempd = {}
            tempd['description'] = job['description']
            tempd['language'] = job['language']
            tempd['rules'] = job['rules']
            community_meta[job['actor_id']].append(tempd)
        else:
            print(job['actor_id'])





In [9]:
community_meta['https://lemmygrad.ml/c/ecomaoism']

[{'description': 'EcoMaoism is the synthesis of Marxism-Leninism-Mao Zedong Thought with radical environmentalist and animal liberation ideologies. We uphold that animals are exploited and deserve the same liberations that the workers would have under communism. We are also against sources of pollution, deforestation, and climate change. We are not western liberals, We are green tankies!',
  'language': 'en',
  'rules': {'rules': {'1': 'EcoMaoism is the synthesis of Marxism-Leninism-Mao Zedong Thought with radical environmentalist and animal liberation ideologies.',
    '2': 'We uphold that animals are exploited and deserve the same liberations that the workers would have under communism.',
    '3': 'We are also against sources of pollution, deforestation, and climate change.',
    '4': 'We are not western liberals, We are green tankies!'}}}]

In [11]:
with open("../data/interim/all_community_meta.json", "w") as jfile:
    json.dump(community_meta, jfile)

TypeError: Object of type JSONDecodeError is not JSON serializable

In [10]:
len(community_meta.keys())

2172

In [ ]:
community_meta['https://lemmy.world/c/mildlyinfuriating']

In [12]:
experiment_meta = community_meta.copy()

In [16]:
exmeta = defaultdict(list)
for c in community_meta.keys():
     if len(community_meta[c]) >= 1:
        for i in range(len(community_meta[c])):
          try:
            if len(community_meta[c][i]['rules'].keys()) >=1:
              exmeta[c].append(community_meta[c][i])
          except:
            print(community_meta[c][i])

        

{'description': 'Environmental and ecological discussion, particularly of things like weather and other natural phenomena (especially if they&#x27;re not breaking news).\n\nSee also our [Nature and Gardening](https://beehaw.org/c/greenspace) community for discussion centered around things like hiking, animals in their natural habitat, and gardening (urban or rural).\n\n---\n\nThis community&#x27;s icon was made by Aaron Schneider, under the [CC-BY-NC-SA 4.0 license](https://creativecommons.org/licenses/by-nc-sa/4.0/).', 'language': 'en', 'rules': JSONDecodeError('Invalid \\uXXXX escape: line 2 column 167 (char 167)')}
{'description': 'Post cute things, don\'t self-doxx (even when you\'re the cutest thing yourself ![soviet-heart](https://lemmy.ml/api/v3/image_proxy?url=https%3A%2F%2Fwww.hexbear.net%2Fpictrs%2Fimage%2Ffbfac5cd-1787-4770-8cc5-a4e748f98f5d.png "emoji soviet-heart")) !\n\nAlso don’t forget to check out [/c/pets](https://hexbear.net/c/pets) for pet posts and [/c/bloomer](htt

In [18]:
with open("../data/interim/final_community_meta.json", "w") as jfile:
    json.dump(exmeta, jfile)

In [17]:
len(exmeta.keys())

2169

In [ ]:
with open("../data/interim/all_community_meta.json", "w") as jfile:
    json.dump(community_meta, jfile)

In [19]:
nrules = []

for c in exmeta.keys():
    for c_ in exmeta[c]:
        nrules.append(len(c_['rules']['rules'].keys()))

In [21]:
from collections import defaultdict, Counter
print(Counter(nrules))

Counter({1: 1237, 3: 304, 4: 259, 5: 255, 2: 185, 6: 183, 7: 141, 8: 120, 9: 82, 10: 66, 12: 19, 0: 17, 14: 10, 11: 9, 15: 4, 18: 3, 19: 3, 21: 3, 99: 2, 13: 2, 16: 1, 17: 1, 32: 1, 37: 1})


In [23]:
## get all unique comments that have description and basic things
modlog_folder = "/storage/lemmymod/lemmymod/modlogs_1733422736.952758/"
community_descriptions = defaultdict(set)
all_comments = set()
comments_with_reason = set()

for instance in tqdm(os.listdir(modlog_folder)): 
    modlog_fpath = f"{modlog_folder}/{instance}/removed_comments.jsonl"
    if not os.path.exists(modlog_fpath): continue
    with open(modlog_fpath, encoding='utf8') as f:
        for entry in map(json.loads, f):
            if 'description' not in entry['community']: continue
            try:
                if len(entry['community']['description']) > 2:
                    community_descriptions[entry['community']['actor_id']].add(entry['community']['description'])
                    all_comments.add(entry['comment']['ap_id'])
                    if "reason" not in entry['mod_remove_comment'] : continue
                    comments_with_reason.add(entry['comment']['ap_id'])
            except:
                pass


  0%|          | 0/865 [00:00<?, ?it/s]

100%|██████████| 865/865 [12:42<00:00,  1.13it/s] 


In [24]:
len(all_comments)

274837

In [25]:
len(comments_with_reason)

109045

In [26]:
len(community_descriptions.keys())

2534